### Dataset and Task Metadata

In [1]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="musk",
    dataset_year="1994",
    domain_str="chemistry & material science",
    # Data Source
    dataset_source="UCI",
    original_dataset_source_download_link="https://doi.org/10.24432/C51608",
    download_description="""
wget https://archive.ics.uci.edu/static/public/75/musk+version+2.zip && unzip musk+version+2.zip clean2.data.Z && uncompress clean2.data.Z && rm musk+version+2.zip && mkdir -p local-data-warehouse/musk && mv clean2.data local-data-warehouse/musk/
""",
    # References
    academic_reference_bibtex="""@article{dietterich1993comparison,
  title={A comparison of dynamic reposing and tangent distance for drug activity prediction},
  author={Dietterich, Thomas and Jain, Ajay and Lathrop, Richard and Lozano-Perez, Tomas},
  journal={Advances in neural information processing systems},
  volume={6},
  year={1993}
}
""",
    academic_reference_bibtex_key="dietterich1993comparison",
    license="CC BY 4.0",
    data_tags=["Non-IID", "Grouped"],
    curation_comments="""
- We rename the molecule IDs to remove the target leakage from the names.
- We drop the conformation name as it leaks information that the real task should not have (the correlation between specific conformations across samples).
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="class",
    problem_type="binary_classification",
    objective_metric_name="roc_auc",
    stratify_on="class",
    group_on="molecule_name",
    group_labels="per_group",
)

## Preprocessing

In [2]:
import pandas as pd
import uuid

df = pd.read_csv(dataset_mold.path / "clean2.data", header=None, names=[
    "molecule_name", "conformation_name", *[f"feature_{i}" for i in range(166)], "class",
])
print("Loaded data shape:", df.shape)

df = df.drop(columns=["conformation_name"])

df["class"] = df["class"].map({0: "non-musk", 1: "musk"})

# Create mapping: molecule -> random string id
mapping = {val: uuid.uuid4().hex[:12] for val in df["molecule_name"].unique()}
df["molecule_name"] = df["molecule_name"].map(mapping)

as_cat_type = ["molecule_name", "class"]
df[as_cat_type] = df[as_cat_type].astype("category")


df = df.sample(frac=1, random_state=42).sort_values(by="molecule_name").reset_index(drop=True)

Loaded data shape: (6598, 169)


## Data Checks

In [3]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    classification=task_mold.is_classification,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)


#### Dataset Overview
Rows: 6,598
Columns: 168
Use sampling: False (sample size: 6,598)
Get row duplicates (staged, merged)...
Using top-10 columns for initial filtering: ['feature_126', 'feature_72', 'feature_33', 'feature_20', 'feature_132', 'feature_55', 'feature_128', 'feature_139', 'feature_59', 'feature_47']
Rows remaining as candidates after top-10 filter: 860 (of 6,598)

#### Duplicate Report
Total duplicate rows: 17 (0.26% of dataset)
Duplicate rows ignoring target: 17 (0.26% of dataset)
Get column duplicates...
Duplicate columns: 0 (0.00% of columns)

Data quality checks completed.


In [4]:
# Sample Rows
df_head

,molecule_name,feature_0,feature_1,feature_2,feature_3,feature_4,feature_5,feature_6,feature_7,feature_8,feature_9,feature_10,feature_11,feature_12,feature_13,feature_14,feature_15,feature_16,feature_17,feature_18,feature_19,feature_20,feature_21,feature_22,feature_23,feature_24,feature_25,feature_26,feature_27,feature_28,feature_29,feature_30,feature_31,feature_32,feature_33,feature_34,feature_35,feature_36,feature_37,feature_38,feature_39,feature_40,feature_41,feature_42,feature_43,feature_44,feature_45,feature_46,feature_47,feature_48,feature_49,feature_50,feature_51,feature_52,feature_53,feature_54,feature_55,feature_56,feature_57,feature_58,feature_59,feature_60,feature_61,feature_62,feature_63,feature_64,feature_65,feature_66,feature_67,feature_68,feature_69,feature_70,feature_71,feature_72,feature_73,feature_74,feature_75,feature_76,feature_77,feature_78,feature_79,feature_80,feature_81,feature_82,feature_83,feature_84,feature_85,feature_86,feature_87,feature_88,feature_89,feature_90,feature_91,feature_92,feature_93,feature_94,feature_95,feature_96,feature_97,feature_98,feature_99,feature_100,feature_101,feature_102,feature_103,feature_104,feature_105,feature_106,feature_107,feature_108,feature_109,feature_110,feature_111,feature_112,feature_113,feature_114,feature_115,feature_116,feature_117,feature_118,feature_119,feature_120,feature_121,feature_122,feature_123,feature_124,feature_125,feature_126,feature_127,feature_128,feature_129,feature_130,feature_131,feature_132,feature_133,feature_134,feature_135,feature_136,feature_137,feature_138,feature_139,feature_140,feature_141,feature_142,feature_143,feature_144,feature_145,feature_146,feature_147,feature_148,feature_149,feature_150,feature_151,feature_152,feature_153,feature_154,feature_155,feature_156,feature_157,feature_158,feature_159,feature_160,feature_161,feature_162,feature_163,feature_164,feature_165,class
0,012a2450ec85,42,-198,-108,-82,-117,-44,37,-83,-13,-29,-186,-166,-141,-202,-195,-302,-49,-75,-144,-69,-35,-16,-49,151,92,29,-47,-109,-100,99,-116,58,-25,39,-132,66,-169,79,-109,50,-184,104,-53,-176,-216,-298,-150,0,-103,-17,-107,-18,1,-130,100,-21,34,-161,-84,18,-101,-154,24,73,-151,57,-166,19,-124,1,-154,-117,-179,-195,-77,-182,-36,-164,-163,-69,5,11,-96,67,-29,41,-165,-50,-79,86,-202,72,-116,-148,29,-39,-129,42,-120,49,-208,135,-158,-178,-200,-186,-169,-30,-147,-203,-138,-32,-4,-16,-44,-9,164,59,51,-78,75,44,47,-107,-106,-9,2,90,-45,-188,18,-138,-47,-139,-160,-2,71,-62,-20,-86,-116,68,61,56,-178,-102,-120,-60,-143,7,-64,-125,-114,-127,99,-116,-237,-75,16,51,127,145,147,-59,-122,55,musk
1,012a2450ec85,42,-197,-148,-92,-117,64,66,-60,-22,5,-183,-101,-52,-194,-201,-302,27,-149,-105,-131,-55,-10,-71,169,96,-28,24,-126,42,73,-116,57,-29,-12,-132,63,-175,-37,-140,30,-179,105,-6,-176,-197,-299,-107,-58,-83,-68,-68,-27,-24,-127,130,-56,87,-153,67,52,-101,-149,24,31,-126,56,-166,-103,-123,-70,-162,-128,-137,-167,-153,-180,4,-161,-103,-54,-55,33,-44,-34,22,34,-121,-172,-9,117,-202,69,-113,-65,28,-41,-115,-1,-155,80,-202,133,-175,-184,-198,-188,-120,-41,-171,-193,-143,-46,-62,-53,-7,35,125,24,76,-9,87,151,149,-108,-105,-9,48,45,12,-187,16,-138,-96,-123,-174,-136,-13,-128,-75,-65,-59,1,58,10,-178,-103,-119,-84,-105,37,-67,-116,-112,-120,99,-73,-240,-165,-191,55,127,143,152,-60,-128,57,musk
2,012a2450ec85,42,-198,-159,-47,-117,-48,43,-81,-29,-65,-210,-123,-130,-201,-174,-302,32,-69,-90,-112,-89,-13,-59,-8,105,10,-23,-84,22,96,-116,57,-29,44,-131,63,-174,1,-140,-40,-184,105,-99,-192,-196,-324,-119,-84,-88,-95,-99,-16,-16,-112,165,-63,51,-151,10,40,-101,-150,24,46,-154,56,-166,-109,-123,-117,-169,-127,-167,-173,-159,-180,4,-180,-101,-68,-83,8,-45,80,4,36,-142,-72,12,69,-202,69,-114,-76,28,-40,-153,40,-155,28,-206,133,-166,-172,-193,-229,-177,-95,-161,-193,-155,-57,-65,-51,25,67,130,59,65,-28,79,146,106,-108,-106,-9,34,90,-1,-187,16,-138,-89,-132,-147,-102,45,-102,-46,-56,-33,-38,17,20,-178,-103,-119,-124,-136,-28,-67,-123,-112,-120,99,-75,-241,-56,-27,55,127,143,152,-60,-1

In [5]:
# Feature Summary
summary

,index,dtype,n_missing,pct_missing,n_unique,examples
0,molecule_name,category,0.0,0.0,102.0,"59d697ad6b2c, bc804ca89aca, 761ce578bca0, da38bb0fcd6f, 73f8d56fb62c, 1edbee62d29c, 56838374a453, b93b59f08f95, bf8e770e691e, a479898a1d69"
1,class,category,0.0,0.0,2.0,"non-musk, musk"
2,feature_0,int64,0.0,0.0,202.0,"44, 43, 35, 36, 46, 51, 48, 37, 47, 57"
3,feature_1,int64,0.0,0.0,260.0,"-198, -194, -199, -193, -192, -195, -196, -197, 86, -191"
4,feature_2,int64,0.0,0.0,221.0,"-145, -144, -112, -19, -22, -111, 31, -62, -23, -146"
5,feature_3,int64,0.0,0.0,257.0,"-76, -77, -69, 28, -70, 29, 33, 131, 32, 152"
6,feature_4,int64,0.0,0.0,129.0,"-117, -116, -115, -113, -112, -111, -114, -108, -110, -109"
7,feature_5,int64,0.0,0.0,358.0,"11, 10, 12, 86, 85, 54, 55, -154, 53, 52"
8,feature_6,int64,0.0,0.0,323.0,"56, 26, 57, -163, 27, -160, -162, -161, -164, -159"
9,feature_7,int64,0.0,0.0,389.0,"-95, -96, -103, 57, 64, -3, -171, -102, 67, -5"


In [6]:
# Numeric Feature Statistics
numeric_stats

,count,mean,std,min,max
feature_0,6598.0,58.945135,53.249007,-31.0,292.0
feature_1,6598.0,-119.128524,90.813375,-199.0,95.0
feature_2,6598.0,-73.146560,67.956235,-167.0,81.0
feature_3,6598.0,-0.628372,80.444617,-114.0,161.0
feature_4,6598.0,-103.533495,64.387559,-118.0,325.0
feature_5,6598.0,18.359806,80.593655,-183.0,200.0
feature_6,6598.0,-14.108821,115.315673,-171.0,220.0
feature_7,6598.0,-1.858290,90.372537,-225.0,320.0
feature_8,6598.0,-86.003031,108.326676,-245.0,147.0
feature_9,6598.0,-44.495756,72.088903,-286.0,231.0


In [7]:
# Categorical Feature Statistics
cat_stats

value  count    pct
column        rank                            
class         1         non-musk   5581  84.59
              2             musk   1017  15.41
molecule_name 1     59d697ad6b2c   1044  15.82
              2     bc804ca89aca   1010  15.31
              3     761ce578bca0    911  13.81
              4     da38bb0fcd6f    383   5.80
              5     73f8d56fb62c    344   5.21

In [8]:
# Target Distribution
target_df

,count,pct
class,,
non-musk,5581,84.59
musk,1017,15.41


## Task Curation

In [9]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(
    dataset=df,
    group_on=task_mold.group_on,
    time_on=task_mold.time_on,
    group_labels=task_mold.group_labels,
)
print(f"Recommended splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

Providing recommendations based on number of groups (102).
Recommended splits: n_repeats=20, n_splits=3, test_size=None


In [10]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry import curation_recommendations

splits = curation_recommendations.get_recommended_grouped_splits(
    dataset=df,
    n_repeats=n_repeats,
    n_splits=n_splits,
    group_on=task_mold.group_on,
    test_size=none_or_test_size,
    stratify_on=task_mold.stratify_on,
    group_labels=task_mold.group_labels,
    show_splits=True,
    target_on=task_mold.target_column_name,
)

splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="We create stratified grouped 20-repeated 3-fold split. This creates ca. 30 group members (500-3000 samples) per test set.",
    splits=splits
)

Using Stratified Grouped splits.
Using label-per-group grouped splits.
Creating index-based splits for 102 groups
Using Stratified IID splits.
Repeat 0, Fold 0:
            Train N: 3560, Test N: 3038
            Target Distribution:
            	Train target distribution: {'non-musk': 0.8328651685393258, 'musk': 0.16713483146067415}
            	Test target distribution: {'non-musk': 0.8610928242264648, 'musk': 0.13890717577353523}
            Group Distribution molecule_name:
            	Train: 68
            	Test: 34
            
Repeat 0, Fold 1:
            Train N: 5571, Test N: 1027
            Target Distribution:
            	Train target distribution: {'non-musk': 0.8571172141446778, 'musk': 0.14288278585532221}
            	Test target distribution: {'non-musk': 0.7848101265822784, 'musk': 0.21518987341772153}
            Group Distribution molecule_name:
            	Train: 68
            	Test: 34
            
Repeat 0, Fold 2:
            Train N: 4065, Test N: 2533
   

## Export

In [11]:
from data_foundry.curation_container import CuratedContainer
curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
 )
curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

Calculating checksum for curated container...
Saving curated container to musk/019d3199-b80d-7a81-913b-6bf76f29a4b8
019d3199-b80d-7a81-913b-6bf76f29a4b8
f4c838aad4e485126298e649a16e1bf7a86d3292376426f363b6c50b3b8e49ec
